# Burnrate Health Data Analysis

Load and analyze Garmin health data from daily JSON files.

In [12]:
import json
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy import stats

sys.path.insert(0, str(Path("..").resolve() / "src"))

# Reload the module to pick up changes
import importlib
from lib.flattener import health_flattener
importlib.reload(health_flattener)
from lib.flattener.health_flattener import HealthFlattener

In [13]:
def load_health_data(health_dir: Path | None = None) -> pd.DataFrame:
    """Load all health JSON files into a single DataFrame.
    
    Args:
        health_dir: Path to health data directory. Defaults to ../data/garmin/health.
    
    Returns:
        DataFrame with flattened health data (stats, heart_rate, body_composition), one row per day.
    """
    if health_dir is None:
        health_dir = Path("../data/garmin/health")
    
    health_files = sorted(health_dir.glob("health_*.json"))
    
    raw_data = []
    for f in health_files:
        with open(f, encoding="utf-8") as fp:
            raw_data.append(json.load(fp))
    
    flattener = HealthFlattener()
    
    # Flatten all three sections
    stats = flattener.flatten_stats(raw_data)
    heart_rate = flattener.flatten_heart_rate(raw_data)
    body_comp = flattener.flatten_body_composition(raw_data)
    
    # Convert to DataFrames
    df_stats = pd.DataFrame(stats)
    df_hr = pd.DataFrame(heart_rate)
    df_bc = pd.DataFrame(body_comp)
    
    # Merge all sections on date
    df = df_stats
    if not df_hr.empty:
        df = df.merge(df_hr, on="date", how="left", suffixes=("", "_hr"))
    if not df_bc.empty:
        df = df.merge(df_bc, on="date", how="left", suffixes=("", "_bc"))
    
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").reset_index(drop=True)

In [14]:
def create_interpolated_data(df: pd.DataFrame, column: str = 'weight', method: str = 'piecewise_polynomial') -> pd.DataFrame:
    """Create interpolated series for a given column.
    
    Args:
        df: Input DataFrame with 'date' and the target column.
        column: The column to interpolate.
        method: Interpolation method.
        
    Returns:
        DataFrame with original and interpolated columns.
    """
    df_interp = df[['date', column]].copy()

    # Store original values
    df_interp[f'{column}_original'] = df_interp[column]

    # Full series (original + filled gaps)
    df_interp[f'{column}_full'] = df_interp[column].interpolate(method=method)
    
    # Interpolated values only (where original was NaN)
    df_interp[f'{column}_interpolated_only'] = df_interp[f'{column}_full'].where(df_interp[f'{column}_original'].isna())
    
    return df_interp

In [15]:
def rolling_trend_regression(series: pd.Series, window: int = 28) -> pd.DataFrame:
    """Calculate rolling linear regression slope and p-value.
    
    Args:
        series: Time series data.
        window: Rolling window size in days.
        
    Returns:
        DataFrame with 'slope' (weekly rate) and 'p_value'.
    """
    slopes = []
    p_values = []
    
    # Iterate through the series with a sliding window
    for i in range(len(series) - window + 1):
        window_data = series.iloc[i:i+window]
        x = np.arange(window)
        y = window_data.values
        
        slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
        
        slopes.append(slope * 7)  # Convert daily change to weekly rate
        p_values.append(p_value)
        
    return pd.DataFrame({
        'slope': slopes,
        'p_value': p_values
    }, index=series.index[window-1:])

In [16]:
raw = load_health_data()
print(f"Loaded {len(raw)} days of health data")
print(f"Date range: {raw['date'].min().date()} to {raw['date'].max().date()}")
raw.head()

Loaded 949 days of health data
Date range: 2023-06-16 to 2026-01-19


,date,userProfileId,totalKilocalories,activeKilocalories,bmrKilocalories,wellnessKilocalories,burnedKilocalories,consumedKilocalories,remainingKilocalories,totalSteps,...,bmi,bodyFat,bodyWater,boneMass,muscleMass,physiqueRating,visceralFat,metabolicAge,startDate,endDate
0,2023-06-16,114046899,3553.0,1345.0,2208.0,3553.0,NaN,NaN,3553.0,8830,...,None,None,None,None,None,None,None,None,2023-06-16,2023-06-16
1,2023-06-17,114046899,2727.0,524.0,2203.0,2727.0,NaN,NaN,2727.0,4814,...,None,None,None,None,None,None,None,None,2023-06-17,2023-06-17
2,2023-06-18,114046899,3325.0,1122.0,2203.0,3325.0,NaN,NaN,3325.0,13058,...,None,None,None,None,None,None,None,None,2023-06-18,2023-06-18
3,2023-06-19,114046899,2534.0,321.0,2213.0,2534.0,NaN,NaN,2534.0,9472,...,None,None,None,None,None,None,None,None,2023-06-19,2023-06-19
4,2023-06-20,114046899,2767.0,554.0,2213.0,2767.0,NaN,NaN,2767.0,13719,...,None,None,None,None,None,None,None,None,2023-06-20,2023-06-20


In [24]:
# Select weight, calorie, and activity-related columns
weight_calorie_cols = [
    'date',
    # Calorie fields
    'totalKilocalories',
    'activeKilocalories',
    'bmrKilocalories',
    'wellnessKilocalories',
    'burnedKilocalories',
    'consumedKilocalories',
    'remainingKilocalories',
    'wellnessActiveKilocalories',
    'netRemainingKilocalories',
    'netCalorieGoal',
    'restingCaloriesFromActivity',
    # Activity fields
    'totalSteps',
    # Weight fields
    'weight',
    'bmi',
    'bodyFat',
    'bodyWater',
    'boneMass',
    'muscleMass',
    'physiqueRating',
    'visceralFat',
    'metabolicAge'
]

# Select only columns that exist in raw
available_cols = [col for col in weight_calorie_cols if col in raw.columns]
df = raw[available_cols].copy()

# Convert weight from grams to kg
df['weight_g'] = df['weight']
df['weight'] = df['weight_g'] / 1000

print(f"Selected {len(available_cols)} columns")
df.head()

Selected 22 columns


,date,totalKilocalories,activeKilocalories,bmrKilocalories,wellnessKilocalories,burnedKilocalories,consumedKilocalories,remainingKilocalories,wellnessActiveKilocalories,netRemainingKilocalories,...,weight,bmi,bodyFat,bodyWater,boneMass,muscleMass,physiqueRating,visceralFat,metabolicAge,weight_g
0,2023-06-16,3553.0,1345.0,2208.0,3553.0,NaN,NaN,3553.0,1345.0,1345.0,...,83.0,None,None,None,None,None,None,None,None,83000.0
1,2023-06-17,2727.0,524.0,2203.0,2727.0,NaN,NaN,2727.0,524.0,524.0,...,82.6,None,None,None,None,None,None,None,None,82600.0
2,2023-06-18,3325.0,1122.0,2203.0,3325.0,NaN,NaN,3325.0,1122.0,1122.0,...,NaN,None,None,None,None,None,None,None,None,NaN
3,2023-06-19,2534.0,321.0,2213.0,2534.0,NaN,NaN,2534.0,321.0,321.0,...,83.4,None,None,None,None,None,None,None,None,83400.0
4,2023-06-20,2767.0,554.0,2213.0,2767.0,NaN,NaN,2767.0,554.0,554.0,...,NaN,None,None,None,None,None,None,None,None,NaN


In [18]:
# Create interpolated weight series using the function
df_interp = create_interpolated_data(df, column='weight')

print(f"Original data points: {df_interp['weight_original'].notna().sum()}")
print(f"Interpolated points: {df_interp['weight_interpolated_only'].notna().sum()}")
print(f"Total with interpolation: {df_interp['weight_full'].notna().sum()}")

# Plot original vs interpolated weight
fig = go.Figure()

# Interpolated values only (red)
fig.add_trace(go.Scatter(
    x=df_interp['date'],
    y=df_interp['weight_full'],
    mode='lines',
    name='Interpolated',
    marker=dict(size=6, color='red', symbol='circle')
))

# Original weight (blue line)
fig.add_trace(go.Scatter(
    x=df_interp['date'],
    y=df_interp['weight_original'],
    mode='lines',
    name='Original',
    line=dict(color='blue', width=2),
    marker=dict(size=4, color='blue')
))

fig.update_layout(
    title='Weight Timeline: Original vs Interpolated',
    xaxis_title='Date',
    yaxis_title='Weight (kg)',
    hovermode='x unified',
    height=500,
    showlegend=True
)

fig.show()

Original data points: 615
Interpolated points: 334
Total with interpolation: 949


In [19]:
# Apply rolling trend regression to interpolated weight (28-day window)
trend_results = rolling_trend_regression(df_interp['weight_full'], window=28)

# Merge results back to df_interp
df_interp = df_interp.merge(trend_results, left_index=True, right_index=True, how='left')

# Preview results where trend starts
df_interp[df_interp['slope'].notna()].head()

,date,weight,weight_original,weight_full,weight_interpolated_only,slope,p_value
27,2023-07-13,83.7,83.7,83.70,NaN,-0.071232,0.398659
28,2023-07-14,82.6,82.6,82.60,NaN,-0.148819,0.083370
29,2023-07-15,NaN,NaN,83.25,83.25,-0.213474,0.006134
30,2023-07-16,NaN,NaN,83.90,83.90,-0.224777,0.003054
31,2023-07-17,NaN,NaN,84.55,84.55,-0.183685,0.028699


In [20]:
# Visualize weight trend (slope)
fig = go.Figure()

# Weekly weight change (slope)
fig.add_trace(go.Bar(
    x=df_interp['date'],
    y=df_interp['slope'],
    name='Weekly Trend (kg/week)',
    marker_color=np.where(df_interp['slope'] < 0, 'green', 'red')
))

# Significance threshold (p < 0.05)
significant = df_interp[df_interp['p_value'] < 0.05]
fig.add_trace(go.Scatter(
    x=significant['date'],
    y=significant['slope'],
    mode='markers',
    name='Significant (p < 0.05)',
    marker=dict(color='black', size=4)
))

fig.update_layout(
    title='28-Day Rolling Weight Trend (Weekly Rate)',
    xaxis_title='Date',
    yaxis_title='Weight Change (kg/week)',
    hovermode='x unified',
    height=400
)

fig.show()

In [25]:
# Create interpolated daily steps series
df_steps = create_interpolated_data(df, column='totalSteps')

print(f"Original steps data points: {df_steps['totalSteps_original'].notna().sum()}")
print(f"Interpolated steps points: {df_steps['totalSteps_interpolated_only'].notna().sum()}")
df_steps.head()

Original steps data points: 949
Interpolated steps points: 0


,date,totalSteps,totalSteps_original,totalSteps_full,totalSteps_interpolated_only
0,2023-06-16,8830,8830,8830,NaN
1,2023-06-17,4814,4814,4814,NaN
2,2023-06-18,13058,13058,13058,NaN
3,2023-06-19,9472,9472,9472,NaN
4,2023-06-20,13719,13719,13719,NaN


In [26]:
# Apply rolling trend regression to interpolated steps (28-day window)
# Only run if we have valid steps data
if df_steps['totalSteps_full'].notna().sum() > 28:
    steps_trend = rolling_trend_regression(df_steps['totalSteps_full'], window=28)

    # Merge results back
    df_steps = df_steps.merge(steps_trend, left_index=True, right_index=True, how='left')

    # Visualize steps trend
    fig = go.Figure()

    # Weekly steps change (slope)
    fig.add_trace(go.Bar(
        x=df_steps['date'],
        y=df_steps['slope'],
        name='Weekly Steps Change (steps/week)',
        marker_color=np.where(df_steps['slope'] < 0, 'orange', 'blue') # Blue for increase, orange for decrease
    ))

    # Significance markers
    significant_steps = df_steps[df_steps['p_value'] < 0.05]
    fig.add_trace(go.Scatter(
        x=significant_steps['date'],
        y=significant_steps['slope'],
        mode='markers',
        name='Significant (p < 0.05)',
        marker=dict(color='black', size=4)
    ))

    fig.update_layout(
        title='28-Day Rolling Daily Steps Trend (Weekly Rate)',
        xaxis_title='Date',
        yaxis_title='Steps Change (steps/week)',
        hovermode='x unified',
        height=400
    )

    fig.show()
else:
    print("No steps data available for trend analysis")